In [44]:
import pandas as pd 
import json
import numpy as np
import spacy
import re
# Tiene funciones para limpiar la info y extraer y guardar los lemas de los mensajes
import funciones

# ETL en la Campaña de Pedidos

### Estructura del caso

Con el propósito de mantener un flujo de trabajo ordenado y mantenible, este cuaderno se apoya en funciones complementarios que evitan la sobrecarga de código en un único archivo:

* **`funciones.py`**: Centraliza las rutinas auxiliares del *pipeline*, integrando funciones dedicadas a la limpieza y normalización del texto de los clientes (desacoplamiento de caracteres especiales, emojis y puntuación) y utilidades para persistir los lemas procesados en archivos `.txt` para su posterior consumo en las fases de NLP.
* **`diccionario.json`**: Almacena de forma externa y estructurada el catálogo de lemas clasificados por categorías (*soporte*, *queja*, *pedido*), permitiendo comparar y categorizar los registros sin necesidad de modificar el código fuente.

Esta arquitectura modular garantiza una ejecución reproducible, facilita las pruebas unitarias y favorece la escalabilidad del análisis.

## Limpieza estructural

In [45]:
df = pd.read_csv('prueba.txt', sep='|', skipinitialspace=True, engine='python',quoting=3)
# print(df.shape) # (3688,26)
# Quita la fila que delimita
df = df.drop(index=0).reset_index(drop=True)
# Quita columnas sin informacion
df = df.dropna(how='all', axis=1) 
# Elimina espacios en las series
df.columns = df.columns.str.strip()
# Quita espacios iniciales y finales en columnas con texto
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()
# print("Limpieza estructural con size:", df.shape) # (3687, 22)
df.head(3)

C:\Users\felim\AppData\Local\Temp\ipykernel_34592\4288273804.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,id,content,account_id,inbox_id,conversation_id,message_type,created_at,updated_at,private,status,...,content_attributes,sender_type,sender_id,additional_attributes,processed_message_content,sentiment,contact_id,sistema_lead_id,conversation_display_id,id_sistema
0,31.888,La conversación fue marcada por Rojas bpo,37,78,2.460,activity,2023-07-28 14:27:58.000,2023-07-28 14:27:58.000,false,sent,...,"""{}""",NaN,NaN,"""{}""",La conversación fue marcada por Rojas bpo,"""{}""",15.090,16.734,1.486,31.888
1,30.276,Gracias por comunicarse con nosotros. 💙¶¶Esper...,37,78,5.355,outgoing,2023-07-26 00:19:49.000,2023-07-26 00:19:51.000,false,delivered,...,"""{}""",User,18,"""{}""",Gracias por comunicarse con nosotros. 💙¶¶Esper...,"""{}""",14.459,16.103,4.265,30.276
2,35.098,"¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó...",37,78,7.226,outgoing,2023-07-29 13:43:11.000,2023-07-29 13:43:12.000,false,failed,...,"""{\""external_error\"": \""(#132000) Number of pa...",User,16,"""{\""template_params\"": {\""name\"": \""lanzamient...","¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó...","""{}""",20.743,22.387,6.006,35.098


## Parsing de JSON

In [46]:
def extraer_datos_json(row):

    error = None
    plantilla = None

    if row.get('status') == 'failed':
        
        # Asegura que la columna content_attributes existe y tiene datos
        if 'content_attributes' in row and pd.notna(row['content_attributes']):
            try:
                #  Hace doble decodificación de JSON para obtener un diccionario, pq comienza con string
                datos_error =  json.loads(json.loads(row['content_attributes']))
                if isinstance(datos_error, dict):
                    claves_error = list(datos_error.keys())                    
                    # Asegura que haya por lo menos una clave
                    if len(claves_error) > 0:
                        error = claves_error[0] 
                        
            except (json.JSONDecodeError, TypeError):
                pass
                
        # Asegura que la columna additional_attributes existe y tiene datos
        if 'additional_attributes' in row and pd.notna(row['additional_attributes']):
            try:
                datos_plantilla =  json.loads(json.loads(row['additional_attributes']))                
                if isinstance(datos_plantilla, dict):
                    params = datos_plantilla.get('template_params', {})
                    if isinstance(params, dict):
                        # Obtenemos la info en la clave 'name'
                        plantilla = params.get('name') 
            except (json.JSONDecodeError, TypeError):
                pass
                
    return pd.Series([error, plantilla])


df[['error_exacto', 'nombre_plantilla']] = df.apply(extraer_datos_json, axis=1)

In [47]:
df[df["status"] == "failed"].head(3)

,id,content,account_id,inbox_id,conversation_id,message_type,created_at,updated_at,private,status,...,sender_id,additional_attributes,processed_message_content,sentiment,contact_id,sistema_lead_id,conversation_display_id,id_sistema,error_exacto,nombre_plantilla
2,35.098,"¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó...",37,78,7.226,outgoing,2023-07-29 13:43:11.000,2023-07-29 13:43:12.000,false,failed,...,16,"""{\""template_params\"": {\""name\"": \""lanzamient...","¡Hola, Johana Andrea Pascuas Perez! 💖¶🚀 ¡Llegó...","""{}""",20.743,22.387,6.006,35.098,external_error,lanzamiento_
3,37.124,"¡Hola, Yeisy Yazmin Fino Galeano! 💖¶🚀 ¡Llegó !...",37,78,2.324,outgoing,2023-07-29 13:56:47.000,2023-07-29 13:56:47.000,false,failed,...,16,"""{\""template_params\"": {\""name\"": \""lanzamient...","¡Hola, Yeisy Yazmin Fino Galeano! 💖¶🚀 ¡Llegó !...","""{}""",17.699,19.343,1.350,37.124,external_error,lanzamiento_
13,36.084,"¡Hola, Liseth Lorena Albanil Furque! 💖¶🚀 ¡Lleg...",37,78,4.347,outgoing,2023-07-29 13:48:59.000,2023-07-29 13:49:00.000,false,failed,...,16,"""{\""template_params\"": {\""name\"": \""lanzamient...","¡Hola, Liseth Lorena Albanil Furque! 💖¶🚀 ¡Lleg...","""{}""",16.265,17.909,3.373,36.084,external_error,lanzamiento_


## Clasificación Básica (NLP ligero)

In [48]:
# Carga el modelo de spaCy para español, desactivando el parser y el NER para mejorar el rendimiento
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
vector_lemas = []

#Extrae lemas de los textos limpiando y separando simbolos
textos_limpios = (
    df["processed_message_content"]
    .dropna()
    .astype(str)
    .apply(funciones.limpiar_y_separar_simbolos)
    .tolist()
)
# Extrae lemas de los textos limpios y los guarda en un archivo
funciones.extraer_lemas(textos_limpios, vector_lemas, nlp)
funciones.guardar_lemas_en_archivo(vector_lemas, "lemas_extraidos.txt")

Archivo guardado exitosamente: lemas_extraidos.txt


In [49]:
# Diccionario de categorías utilizando LEMAS (verbos en infinitivo y palabras raiz)
with open("diccionario.json", "r", encoding="utf-8") as archivo:
    diccionario_lemas = json.load(archivo)
# funcion usando spaCy para clasificar el texto en categorías según los lemas
def clasificar_nlp(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return 'Sin texto'
        
    doc = nlp(str(texto).lower())
    
    # Guardamos tanto el lema (raíz) como el texto original exacto
    terminos_cliente = []
    for token in doc:
        if not token.is_punct and not token.is_stop:
            terminos_cliente.append(token.lemma_)      
            terminos_cliente.append(token.text.lower()) 
    
    for categoria, palabras_clave in diccionario_lemas.items():
        if any(termino in palabras_clave for termino in terminos_cliente):
            # Si alguna palabra del mensaje está en la lista de claves de la categoría, devuelve la categoría
            return categoria
    # Si no hay coincidencias, devuelve 'General' 
    return 'General'
# Aplicar la clasificación inteligente a los mensajes entrantes
df['categoria'] = np.where(
    df['message_type'] == 'incoming',
    df['content'].apply(clasificar_nlp),
    None 
)

In [50]:
df[df["message_type"]=='incoming'].head(4)

,id,content,account_id,inbox_id,conversation_id,message_type,created_at,updated_at,private,status,...,additional_attributes,processed_message_content,sentiment,contact_id,sistema_lead_id,conversation_display_id,id_sistema,error_exacto,nombre_plantilla,categoria
5,49.903,En una factura me cobraon algo q no llego,37,78,6.301,incoming,2023-07-30 15:50:18.000,2023-07-30 15:50:18.000,false,sent,...,"""{}""",En una factura me cobraon algo q no llego,"""{}""",19.798,21.442,5.162,49.903,None,None,queja
6,51.449,Si,37,78,9.415,incoming,2023-07-30 22:31:12.000,2023-07-30 22:31:12.000,false,sent,...,"""{}""",Si,"""{}""",13.702,15.346,8.195,51.449,None,None,General
11,41.906,Estoy inconforme con un premio que pedí,37,78,2.158,incoming,2023-07-29 19:19:07.000,2023-07-29 19:19:07.000,false,sent,...,"""{}""",Estoy inconforme con un premio que pedí,"""{}""",20.202,21.846,1.184,41.906,None,None,queja
15,54.655,Ingresarlo,37,78,1.191,incoming,2023-08-01 19:29:32.000,2023-08-01 19:29:32.000,false,sent,...,"""{}""",Ingresarlo,"""{}""",17.108,18.752,217,54.655,None,None,General


In [51]:
import csv
df = df.replace(r'\n|\r', ' ', regex=True)

# Exportamos protegiendo las comillas
df.to_csv(
    'prueba_procesado.csv', 
    index=False, 
    encoding='utf-8', 
    quoting=csv.QUOTE_ALL, # Obliga a poner comillas alrededor de cada campo
    escapechar='\\'        # Protege las comillas internas
)
print("Archivo exportado con formato seguro.") 

Archivo exportado con formato seguro.
